# Module 4: Rossmann Store Sales Time-Series Forecasting (1,115 Stores)

Welcome to **Module 4**! In this lab, you will forecast daily sales for 1,115 Rossmann drug stores across Germany (`rossmann_train.csv`, `rossmann_store.csv`, `rossmann_test.csv`).

### Forecasting Brief:
> *"Predict daily sales for each store, six weeks into the future — and tell us how much we can trust the numbers."*

---

### Assignment Tasks Overview:
1. **Task 1**: Load train, store, and test CSV datasets; merge store metadata.
2. **Task 2**: Filter closed stores (`Open == 0`) and clean zero-sales days.
3. **Task 3**: Time-based Data Splitting (Avoid future-to-past data leakage!).
4. **Task 4**: Feature Engineering (Lagged sales, promo indicators, competitor age).
5. **Task 5**: Model Training & RMSPE Evaluation against Baseline moving average.

In [2]:
# Task 1: Load Rossmann Datasets & Merge Metadata
import pandas as pd
import numpy as np

# Load datasets
train = pd.read_csv('rossmann_train.csv', low_memory=False)
store = pd.read_csv('rossmann_store.csv')

# Merge store metadata
df = pd.merge(train, store, on='Store', how='left')
df['Date'] = pd.to_datetime(df['Date'])

# Filter open stores with positive sales
df_open = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()

print(f"Total Open Sales Records: {len(df_open):,}")
df_open[['Date', 'Store', 'Sales', 'Customers', 'StoreType', 'Promo']].head(10)

Total Open Sales Records: 844,338



<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Date</th>
      <th>Store</th>
      <th>Sales</th>
      <th>Customers</th>
      <th>StoreType</th>
      <th>Promo</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>2015-07-31</td>
      <td>1</td>
      <td>5263</td>
      <td>555</td>
      <td>c</td>
      <td>1</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2015-07-31</td>
      <td>2</td>
      <td>6064</td>
      <td>625</td>
      <td>a</td>
      <td>1</td>
    </tr>
    <tr>
      <th>2</th>
      <td>2015-07-31</td>
      <td>3</td>
      <td>8314</td>
      <td>821</td>
      <td>a</td>
      <td>1</td>
    </tr>
    <tr>
      <th>3</th>
      <td>2015-07-31

In [3]:
# Task 2: Clean closed stores and zero-sales days

df_clean = df[(df['Open'] == 1) & (df['Sales'] > 0)].copy()

print("Records after cleaning:", len(df_clean))
print("Closed stores remaining:", (df_clean['Open'] == 0).sum())
print("Zero-sales days remaining:", (df_clean['Sales'] == 0).sum())

Records after cleaning: 844338
Closed stores remaining: 0
Zero-sales days remaining: 0



In [6]:
import pandas as pd

train = pd.read_csv('rossmann_train.csv', low_memory=False)
store = pd.read_csv('rossmann_store.csv')

df = pd.merge(train, store, on='Store', how='left')

df['Date'] = pd.to_datetime(df['Date'])

df_clean = df[
    (df['Open'] == 1) &
    (df['Sales'] > 0)
].copy()

print("df_clean created successfully")
print("Records:", len(df_clean))

df_clean created successfully
Records: 844338



In [7]:


# Task 4: Feature Engineering

# Sort by store and date
df_clean = df_clean.sort_values(['Store', 'Date']).copy()

# 1. Previous day's sales for each store
df_clean['Sales_Lag_1'] = df_clean.groupby('Store')['Sales'].shift(1)

# 2. Previous week's sales for each store
df_clean['Sales_Lag_7'] = df_clean.groupby('Store')['Sales'].shift(7)

# 3. Competitor age in months
df_clean['CompetitionOpenSinceMonth'] = (
    df_clean['CompetitionOpenSinceMonth'].fillna(0)
)

df_clean['CompetitionOpenSinceYear'] = (
    df_clean['CompetitionOpenSinceYear'].fillna(0)
)

df_clean['CompetitorAgeMonths'] = np.where(
    (df_clean['CompetitionOpenSinceYear'] > 0) &
    (df_clean['CompetitionOpenSinceMonth'] > 0),
    (df_clean['Date'].dt.year - df_clean['CompetitionOpenSinceYear']) * 12
    + (df_clean['Date'].dt.month - df_clean['CompetitionOpenSinceMonth']),
    0
)

# Remove rows where lagged sales are unavailable
df_features = df_clean.dropna(
    subset=['Sales_Lag_1', 'Sales_Lag_7']
).copy()

print("Feature-engineered dataset shape:", df_features.shape)

df_features[
    ['Store', 'Date', 'Sales', 'Sales_Lag_1',
     'Sales_Lag_7', 'Promo', 'CompetitorAgeMonths']
].head(10)

Feature-engineered dataset shape: (836533, 21)



In [8]:
# Recreate Task 4 feature-engineered dataset

df_clean = df_clean.sort_values(['Store', 'Date']).copy()

df_clean['Sales_Lag_1'] = (
    df_clean.groupby('Store')['Sales'].shift(1)
)

df_clean['Sales_Lag_7'] = (
    df_clean.groupby('Store')['Sales'].shift(7)
)

df_clean['CompetitionOpenSinceMonth'] = (
    df_clean['CompetitionOpenSinceMonth'].fillna(0)
)

df_clean['CompetitionOpenSinceYear'] = (
    df_clean['CompetitionOpenSinceYear'].fillna(0)
)

df_clean['CompetitorAgeMonths'] = np.where(
    (df_clean['CompetitionOpenSinceYear'] > 0) &
    (df_clean['CompetitionOpenSinceMonth'] > 0),
    (df_clean['Date'].dt.year - df_clean['CompetitionOpenSinceYear']) * 12
    + (df_clean['Date'].dt.month - df_clean['CompetitionOpenSinceMonth']),
    0
)

df_features = df_clean.dropna(
    subset=['Sales_Lag_1', 'Sales_Lag_7']
).copy()

print("Feature-engineered dataset shape:", df_features.shape)

Feature-engineered dataset shape: (836533, 21)



In [13]:
# Task 5: Model Training & RMSPE Evaluation

import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor

# ML model
model = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Predict
ml_predictions = model.predict(X_test)

print("ML model trained successfully!")
print("Number of predictions:", len(ml_predictions))

ML model trained successfully!
Number of predictions: 41396



In [14]:
# Task 5: RMSPE Evaluation

# Moving Average baseline
moving_average_predictions = test_ml['Sales_Lag_7'].values

# RMSPE function
def rmspe(actual, predicted):
    return np.sqrt(
        np.mean(
            ((actual - predicted) / actual) ** 2
        )
    )

# Calculate RMSPE
ml_rmspe = rmspe(
    y_test.values,
    ml_predictions
)

baseline_rmspe = rmspe(
    y_test.values,
    moving_average_predictions
)

print("=== RMSPE Comparison ===")
print("ML Model RMSPE:", round(ml_rmspe, 4))
print("Moving Average RMSPE:", round(baseline_rmspe, 4))

if ml_rmspe < baseline_rmspe:
    print("\nML Model performs better than Moving Average.")
else:
    print("\nMoving Average performs better than ML Model.")

=== RMSPE Comparison ===
ML Model RMSPE: 0.1342
Moving Average RMSPE: 0.4846

ML Model performs better than Moving Average.

